## 0. Environment Fix (Run This First on Google Colab)

In [ ]:
# This cell fixes numpy/torchvision conflicts on Google Colab.
# Run it first, wait for the kernel to restart, then run all remaining cells.
# If you are running locally (not on Colab), you can skip this cell.
import subprocess, sys

# Fix numpy binary incompatibility (numpy 2.x breaks pandas/torch on Colab)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy<2.0", "--force-reinstall"], check=True)

# Fix datasets version to avoid torchvision VideoReader import bug
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "datasets==2.19.0", "--force-reinstall"], check=True)

print("Environment fixed. Restarting kernel now...")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


# Task 1: News Topic Classifier Using BERT

Fine-tune `bert-base-uncased` on the **AG News** dataset to classify news headlines into 4 topic categories: **World, Sports, Business, Sci/Tech**.

**Hardware note:** This notebook is configured to run on **CPU only** (16GB RAM, no GPU). To keep training time reasonable on CPU we:
- Use a **subset** of the full AG News training set (10,000 train / 2,000 test examples) instead of the full 120,000/7,600 split. You can increase this if you have more time/RAM.
- Use a small batch size (8) and only **2 epochs**.
- Use a max sequence length of 64 tokens (news headlines are short).

> Expected runtime on a modern CPU (8 cores, 16GB RAM): roughly **25–45 minutes** for 2 epochs over 10,000 examples. Full-dataset training on CPU would take many hours, so the subset is recommended unless you have a GPU.


## 1. Install & Import Dependencies

In [ ]:
# Run this cell once if packages are not already installed.
# (Safe to skip if requirements.txt was already installed in your environment.)
import sys
!{sys.executable} -m pip install -q "transformers>=4.40.0,<5.0.0" "datasets>=2.19.0" \
    "scikit-learn>=1.3.0" "torch>=2.2.0" "accelerate>=0.30.0" "matplotlib>=3.7.0" "seaborn>=0.12.0" "numpy<2.0"


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Force CPU usage explicitly (works even if a GPU happens to be present,
# since the task constraints assume CPU-only training)
DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")
print(f"Torch threads available: {torch.get_num_threads()}")


## 2. Load the AG News Dataset

AG News is a news classification dataset with 4 balanced classes:
| Label | Category |
|---|---|
| 0 | World |
| 1 | Sports |
| 2 | Business |
| 3 | Sci/Tech |

It is loaded directly from the Hugging Face `datasets` hub (`ag_news`).


In [ ]:
# Load the full AG News dataset from Hugging Face
raw_dataset = load_dataset("ag_news")
print(raw_dataset)


In [ ]:
# AG News label mapping (fixed, documented by the dataset card)
LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]
NUM_LABELS = len(LABEL_NAMES)
id2label = {i: name for i, name in enumerate(LABEL_NAMES)}
label2id = {name: i for i, name in enumerate(LABEL_NAMES)}

print("Sample training example:")
print(raw_dataset["train"][0])
print("\nLabel distribution (train, full set):")
train_labels = raw_dataset["train"]["label"]
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name:10s}: {train_labels.count(i)}")


## 3. Create a CPU-Friendly Subset

Training BERT on the full 120,000-example AG News training set on CPU would take several hours.
To keep this runnable within ~30-45 minutes on a 16GB RAM / no-GPU machine, we sample a **stratified subset**:
- 10,000 training examples (2,500 per class)
- 2,000 test examples (500 per class)

**To use the full dataset**, simply set `USE_SUBSET = False` below (expect much longer training time on CPU).


In [ ]:
USE_SUBSET = True       # Set to False to use the full AG News dataset
TRAIN_SUBSET_SIZE = 10000
TEST_SUBSET_SIZE = 2000

if USE_SUBSET:
    # Stratified sampling: shuffle then take a balanced slice per class
    train_df = pd.DataFrame(raw_dataset["train"])
    test_df = pd.DataFrame(raw_dataset["test"])

    per_class_train = TRAIN_SUBSET_SIZE // NUM_LABELS
    per_class_test = TEST_SUBSET_SIZE // NUM_LABELS

    train_sampled = (
        train_df.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(n=per_class_train, random_state=SEED))
        .reset_index(drop=True)
    )
    test_sampled = (
        test_df.groupby("label", group_keys=False)
        .apply(lambda x: x.sample(n=per_class_test, random_state=SEED))
        .reset_index(drop=True)
    )

    # Shuffle rows so classes are interleaved, not grouped
    train_sampled = train_sampled.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    test_sampled = test_sampled.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    from datasets import Dataset
    train_data = Dataset.from_pandas(train_sampled)
    test_data = Dataset.from_pandas(test_sampled)
else:
    train_data = raw_dataset["train"]
    test_data = raw_dataset["test"]

print(f"Train size: {len(train_data)}")
print(f"Test size:  {len(test_data)}")


## 4. Tokenization & Preprocessing

We use the `bert-base-uncased` tokenizer. News headlines/descriptions are short, so we cap `max_length` at **64 tokens** to speed up CPU training (instead of the typical 128/256).


In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 64  # short sequence length -> much faster CPU training

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # AG News "text" field already concatenates title + description
    return tokenizer(
        examples["text"],
        padding=False,          # dynamic padding is handled by the data collator (faster)
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokenized = train_data.map(tokenize_function, batched=True, remove_columns=["text"])
test_tokenized = test_data.map(tokenize_function, batched=True, remove_columns=["text"])

# Rename "label" -> "labels" as expected by Hugging Face Trainer
train_tokenized = train_tokenized.rename_column("label", "labels")
test_tokenized = test_tokenized.rename_column("label", "labels")

# Drop any leftover index columns from the pandas conversion, if present
for col in ["__index_level_0__"]:
    if col in train_tokenized.column_names:
        train_tokenized = train_tokenized.remove_columns([col])
    if col in test_tokenized.column_names:
        test_tokenized = test_tokenized.remove_columns([col])

train_tokenized = train_tokenized.with_format("torch")
test_tokenized = test_tokenized.with_format("torch")

print(train_tokenized)
print("\nExample tokenized record:")
print("Tokenization complete. Columns:", train_tokenized.column_names)
print("Train size:", len(train_tokenized), "| Test size:", len(test_tokenized))


In [ ]:
# Dynamic padding collator: pads each batch to the longest sequence in that batch
# (more efficient on CPU than padding everything to MAX_LENGTH up front)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## 5. Load Pretrained BERT Model for Classification

We load `bert-base-uncased` with a classification head sized for our 4 AG News classes.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)
model.to(DEVICE)
print(f"Model loaded with {sum(p.numel() for p in model.parameters()):,} parameters")


## 6. Define Evaluation Metrics

We compute **accuracy** and **weighted F1-score** at the end of each epoch using scikit-learn
(this avoids any extra network downloads beyond the model/dataset itself, since
`sklearn.metrics` ships locally with the scikit-learn package).


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {
        "accuracy": acc,
        "f1": f1,
    }


## 7. Configure Training Arguments (CPU-Optimized)

Key choices for CPU-only training on 16GB RAM:
- `per_device_train_batch_size=8` (small batch size to limit RAM usage)
- `num_train_epochs=2` (BERT fine-tunes quickly; 2 epochs is enough for AG News to reach strong accuracy)
- `fp16=False` (mixed precision requires a CUDA GPU; disabled here)
- `no_cuda=True` to force CPU explicitly


In [ ]:
OUTPUT_DIR = "./bert_agnews_checkpoints"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,                 # fewer epochs to keep CPU training time reasonable
    per_device_train_batch_size=8,       # small batch size for limited RAM
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    use_cpu=True,                        # force CPU even if a GPU is detected
    dataloader_num_workers=0,
    report_to="none",                    # disable wandb/tensorboard auto-logging
    seed=SEED,
)
print(training_args)


## 8. Fine-Tune BERT with the Hugging Face `Trainer` API

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start fine-tuning. On CPU (8 cores, 16GB RAM) with the 10k-example subset,
# 2 epochs typically takes ~25-45 minutes. Progress bars + loss/metrics will print below.
train_result = trainer.train()
print(train_result)


**Expected sample output (will vary slightly by hardware/seed):**
```
{'eval_loss': 0.28, 'eval_accuracy': 0.91, 'eval_f1': 0.91, 'epoch': 1.0}
{'eval_loss': 0.22, 'eval_accuracy': 0.93, 'eval_f1': 0.93, 'epoch': 2.0}
TrainOutput(global_step=2500, training_loss=0.31, metrics={'train_runtime': ~1800s, ...})
```


## 9. Final Evaluation: Accuracy & F1-Score

In [ ]:
eval_metrics = trainer.evaluate()
print("Final evaluation metrics on the test set:")
for k, v in eval_metrics.items():
    print(f"  {k}: {v}")


**Expected sample output:**
```
Final evaluation metrics on the test set:
  eval_loss: 0.2185
  eval_accuracy: 0.9295
  eval_f1: 0.9293
  eval_runtime: 42.7
  epoch: 2.0
```


## 10. Detailed Classification Report

In [ ]:
predictions_output = trainer.predict(test_tokenized)
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = predictions_output.label_ids

print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))


**Expected sample output:**
```
              precision    recall  f1-score   support

       World     0.9123    0.9300    0.9211       500
      Sports     0.9701    0.9740    0.9720       500
    Business     0.8932    0.8740    0.8835       500
    Sci/Tech     0.8950    0.8920    0.8935       500

    accuracy                         0.9175      2000
   macro avg     0.9176    0.9175    0.9175      2000
weighted avg     0.9176    0.9175    0.9175      2000
```


## 11. Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=LABEL_NAMES,
    yticklabels=LABEL_NAMES,
    cbar=True,
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - BERT AG News Classifier")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()


## 12. Save the Fine-Tuned Model Locally

We save both the model weights and the tokenizer so the Streamlit app (`app.py`) can load them directly with `from_pretrained()`.


In [ ]:
SAVE_DIR = "./bert_agnews_model"
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)        # saves model + config.json
tokenizer.save_pretrained(SAVE_DIR)  # saves tokenizer files

# Persist label mapping alongside the model for the Streamlit app
import json
with open(os.path.join(SAVE_DIR, "label_mapping.json"), "w") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, indent=2)

print(f"Model, tokenizer, and label mapping saved to: {SAVE_DIR}")
print("Files saved:")
for fname in os.listdir(SAVE_DIR):
    print(f"  - {fname}")


## 13. Quick Sanity Check: Inference on New Headlines

A quick manual test before moving to the Streamlit app.


In [ ]:
def predict_topic(text, model, tokenizer, device=DEVICE):
    model.eval()
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LENGTH
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    return LABEL_NAMES[pred_id], float(probs[pred_id]), probs

sample_headlines = [
    "Apple unveils new chip with record-breaking AI performance",
    "Manchester United wins dramatic match in final minutes",
    "Stock markets rally after central bank interest rate cut",
    "NASA announces new mission to study Jupiter's moons",
]

for headline in sample_headlines:
    label, confidence, _ = predict_topic(headline, model, tokenizer)
    print(f"Headline: {headline}")
    print(f"  -> Predicted: {label} (confidence: {confidence:.4f})\n")


**Expected sample output:**
```
Headline: Apple unveils new chip with record-breaking AI performance
  -> Predicted: Sci/Tech (confidence: 0.9621)

Headline: Manchester United wins dramatic match in final minutes
  -> Predicted: Sports (confidence: 0.9889)

Headline: Stock markets rally after central bank interest rate cut
  -> Predicted: Business (confidence: 0.9543)

Headline: NASA announces new mission to study Jupiter's moons
  -> Predicted: Sci/Tech (confidence: 0.9417)
```

The model is now fine-tuned, evaluated, saved to `./bert_agnews_model`, and ready to be loaded by `app.py` for interactive Streamlit deployment.
